# LangChain 表达式语言（LCEL）学习笔记

把每一步看作一个有输入和输出的函数。LCEL 让提示词模板、聊天模型、输出解析器和普通 Python 函数都能接到同一条链上。**`A | B` 表示把 A 的输出交给 B。** 写下这条表达式是在组装流程；调用 `invoke`、`stream` 等方法才开始执行。

这节课围绕一个「校园图书馆小助手」展开。选择 `langchain_env` 内核，从上往下运行。每节标有 **模型** 或 **本地**；本地例子可反复运行。默认完整运行会调用模型 12 次（按成功且无重试计算）。只阅读已保存的输出不会产生 API 用量；第 9 节的交互聊天默认关闭。

| 想解决的问题 | 对应章节 |
|---|---|
| 把提问变成文字回复，弄清每步输入、输出 | 1 串联 · 2 接入 Python |
| 同一份材料生成多种结果，不同问题走不同流程 | 3 并行与合并 · 4 条件分支 |
| 处理多份输入、从异步程序调用、边生成边显示 | 5 批量与异步 · 6 流式 |
| 提取字段、依据资料回答、连续追问 | 7 结构化输出 · 8 检索增强 · 9 多轮聊天 |
| 处理临时故障，定位一条链在哪一步出错 | 10 重试与备用流程 · 11 检查和调试 |

## 准备：连接模型

沿用同目录 `.env` 的密钥和代理。以下例子只需要短回复，每次生成上限设为 512 tokens。`bind` 将固定调用参数绑定到模型，此处给这些简单练习选择无推理模式。

In [1]:
from dotenv import dotenv_values
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableBranch, RunnableLambda, RunnableParallel, RunnablePassthrough,
)

config = dotenv_values(".env")
if not config.get("OPENAI_API_KEY") or not config.get("OPENAI_PROXY"):
    raise ValueError("请在本目录的 .env 中填写 OPENAI_API_KEY 和 OPENAI_PROXY。")

model = ChatOpenAI(
    model="gpt-6-luna",
    api_key=config["OPENAI_API_KEY"],
    openai_proxy=config["OPENAI_PROXY"],
    base_url="https://api.openai.com/v1",
    use_responses_api=True,
    max_completion_tokens=512,
).bind(reasoning={"effort": "none"})

text_parser = StrOutputParser()

## 1. 串联：从一个词得到一段解释 · 模型

`dict → ChatPromptValue → AIMessage → str`。第一步用字典填模板，第二步让模型回答，第三步取出回答文本。用 `|` 连起来得到的是 `RunnableSequence`。

`partial` 给模板预填固定变量。这里预填读者身份，调用方只需提供 `term`。

In [2]:
term_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是校园图书馆员。用面向{audience}的中文回答，最多两句话。"),
    ("human", "请解释“{term}”，举一个小例子。"),
]).partial(audience="第一次来借书的同学")

term_chain = term_prompt | model | text_parser
example = {"term": "续借"}

# 只运行第一步，看看模型将收到哪些角色和内容；这行只在本地填模板。
print(term_prompt.invoke(example).to_string())
print("回答：", term_chain.invoke(example))

System: 你是校园图书馆员。用面向第一次来借书的同学的中文回答，最多两句话。
Human: 请解释“续借”，举一个小例子。
回答： “续借”是指在借阅期限快到或已到时，向图书馆申请延长还书时间。比如你借的书原定周五归还，办理续借后可以多借一段时间再还。


## 2. 接入 Python：让链直接接收字符串 · 本地

网页输入框通常给你一个字符串，而上面的模板需要含 `term` 的字典。`RunnableLambda` 把清洗和转换输入的普通函数包装成可以拼接的步骤。先单独测这一步，再接入已有的链。

In [3]:
def prepare_term(raw_text):
    term = raw_text.strip()
    if not term:
        raise ValueError("请输入一个词。")
    return {"term": term}

prepare = RunnableLambda(prepare_term)
reader_chain = prepare | term_chain   # 对外接收 str，对内复用第 1 节的链
print(prepare.invoke("  预约借书  "))   # 这里只查看送入模板的字典

{'term': '预约借书'}


## 3. 并行与合并：把一条通知变成摘要和待办 · 模型

`RunnableParallel` 会把**同一份输入**发给每个分支，结果放进以分支名为键的字典。摘要和待办互不依赖，可以同时生成；这里会调用模型两次。`RunnablePassthrough()` 原样保留输入，`assign` 再利用已有结果给字典增加字段。

**读懂常见简写**　在管道里，`{"summary": summary_chain, "todo": todo_chain} | next_step` 等价于先运行 `RunnableParallel(summary=summary_chain, todo=todo_chain)`，再把结果字典交给 `next_step`。

In [4]:
notice_text = "周五下午三点图书馆举行读书会，参加者请在周四前报名，并自带一本书。"
summary_prompt = ChatPromptTemplate.from_template("把这条通知概括为一句话：{notice}")
todo_prompt = ChatPromptTemplate.from_template("从这条通知中提取读者待办，每项一行：{notice}")

notice_parallel = RunnableParallel(
    original=RunnablePassthrough(),
    summary=summary_prompt | model | text_parser,
    todo=todo_prompt | model | text_parser,
)
# original 保留整个输入字典；assign 能读取三个分支合并后的结果。
notice_chain = notice_parallel | RunnablePassthrough.assign(
    source_chars=lambda result: len(result["original"]["notice"])
)
notice_result = notice_chain.invoke({"notice": notice_text})
print("原文字符数：", notice_result["source_chars"])
print("摘要：", notice_result["summary"])
print("待办：", notice_result["todo"])

原文字符数： 33
摘要： 周五下午三点图书馆举行读书会，请参加者于周四前报名并自带一本书。
待办： - 周四前报名参加读书会。
- 自带一本书。
- 周五下午三点前往图书馆参加读书会。


## 4. 条件分支：有固定答案就直接查 · 本地 + 模型

`RunnableBranch` 按顺序检查条件，只执行第一个命中的分支；都没有命中时执行最后的默认分支。与上一节比较：同一条输入在这里只走一条路径。馆藏词表能回答的提问无需调用模型。

In [5]:
glossary = {"索书号": "图书在书架上的定位编号。"}
lookup = RunnableLambda(lambda term: glossary[term])
route = RunnableBranch(
    (lambda term: term in glossary, lookup),
    reader_chain,   # 默认路径使用第 2 节的字符串输入链
)
print("词表回答：", route.invoke("索书号"))
print("模型回答：", route.invoke("馆际互借"))

词表回答： 图书在书架上的定位编号。
模型回答： “馆际互借”是指本校图书馆没有你要的书时，帮你向其他图书馆借来。比如你想看的书本馆没有，图书馆可以联系附近高校图书馆借给你。


## 5. 批量与异步：给两个借阅术语各写一份解释 · 模型 + 本地

上一节的并行是「一个输入，多个分支」；`batch` 是「多个输入，同一条链」。结果列表与输入顺序对应。这里两个词各请求一次模型。`max_concurrency` 控制同时运行的任务数；token 上限在模型配置中单独设置。

In [6]:
terms = ["闭架借阅", "馆藏目录"]
answers = reader_chain.batch(terms, config={"max_concurrency": 2})
for term, answer in zip(terms, answers):
    print(f"{term}：{answer}")

闭架借阅：“闭架借阅”是指图书不由读者自行进入书库挑选，而是由馆员根据书目或索书号取出后借给读者。比如，你在目录里查到一本书并提交借阅请求，馆员就会帮你从书库取来。
馆藏目录：“馆藏目录”是图书馆用来查询馆藏书籍信息的清单，可以查到书名、作者、索书号和是否在馆等。比如搜索《小王子》，目录显示它在二楼儿童文学区、索书号为 I565.88/123，就能按索书号去找书。


异步程序里，单条用 `await chain.ainvoke(...)`，批量用 `await chain.abatch(...)`。Jupyter 支持直接写 `await`。下面用本地步骤演示；把 `prepare` 换成 `reader_chain` 就会调用模型。

In [7]:
one = await prepare.ainvoke("  续借 ")
many = await prepare.abatch([" 书签 ", " 目录 "], config={"max_concurrency": 2})
print("异步单条：", one)
print("异步批量：", many)

异步单条： {'term': '续借'}
异步批量： [{'term': '书签'}, {'term': '目录'}]


**多步生成：先分项解释，再写总览。** 上面把同一个任务应用到每个术语，相当于 map；下面把这些产物合并后再请模型整理，相当于 reduce。这也是长文分段摘要的常见结构。复用 `answers`，这里只增加一次调用。

In [8]:
overview_prompt = ChatPromptTemplate.from_template(
    "依据下面两条解释，写一段不超过 80 字的新生借书导览。\n{notes}"
)
notes = "\n".join(f"{term}：{answer}" for term, answer in zip(terms, answers))
overview = (overview_prompt | model | text_parser).invoke({"notes": notes})
print(overview)

新生借书先查馆藏目录，了解书名、索书号和在馆情况。若为闭架借阅，提交借阅请求后由馆员取书；开放书库则可按索书号自行找书。


## 6. 流式：让读者边等边看 · 模型

同一条 `reader_chain`，`invoke` 返回完整字符串，`stream` 逐段产出字符串。可以一边显示一边收集，结束后拼回完整答案。一次流式生成只发起一次模型调用，片段数量由服务端决定。

In [9]:
chunks = []
for chunk in reader_chain.stream("电子借阅证"):
    print(chunk, end="", flush=True)
    chunks.append(chunk)
full_answer = "".join(chunks)
print(f"\n共收到 {len(full_answer)} 个字符。")

“电子借阅证”是保存在手机或校园平台上的数字版借书证，可用来办理借书、还书等业务。比如，你在图书馆出示手机上的电子借阅证，工作人员扫码后就能为你办理借书。
共收到 77 个字符。


异步流式的对应写法是 `async for chunk in reader_chain.astream("电子借阅证")`。想要逐段显示时，把需要等到全文才能完成的处理放在显示之后；整段摘要这类步骤会让下游等到上游结束。

## 7. 结构化输出：把读者留言填进借书表 · 模型

界面要展示一句话时，用 `StrOutputParser`。表单需要姓名、书名和正整数天数，这里把链尾换成 `PydanticOutputParser`。字段定义也会生成格式说明，插入提示词供模型遵循。

In [10]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class LoanRequest(BaseModel):
    borrower: str
    title: str
    days: int = Field(gt=0)

loan_parser = PydanticOutputParser(pydantic_object=LoanRequest)
loan_prompt = ChatPromptTemplate.from_messages([
    ("system", "按给定格式提取借书申请。只返回 JSON。\n{format_rules}"),
    ("human", "{note}"),
]).partial(format_rules=loan_parser.get_format_instructions())

loan_chain = loan_prompt | model | loan_parser
loan = loan_chain.invoke({"note": "小林想借《西游记》，借 7 天。"})
print(type(loan).__name__, loan.model_dump())
print("借阅天数加一：", loan.days + 1)

LoanRequest {'borrower': '小林', 'title': '西游记', 'days': 7}
借阅天数加一： 8


## 8. 检索增强（RAG）：根据本馆规则回答，并显示出处 · 本地 + 模型

借阅规则因馆而异。先在两条虚构馆规中用关键词检索，再把命中的条目和原问题交给模型。真实知识库也可以接在同一个位置。这里特意保留来源字段，方便读者核对；查不到的题走固定答复。

In [11]:
rules = {
    "续借": {"source": "借阅须知 §1", "text": "到期前可续借一次，延长 14 天。"},
    "遗失": {"source": "借阅须知 §2", "text": "请先到服务台登记遗失的图书。"},
}

def retrieve(question):
    for keyword, entry in rules.items():
        if keyword in question:
            return entry
    return None

retrieve_step = RunnableParallel(question=RunnablePassthrough(), entry=RunnableLambda(retrieve))
grounded_prompt = ChatPromptTemplate.from_template(
    "请仅根据馆规回答；资料不足就说明。\n馆规：{context}\n问题：{question}"
)
answer_from_entry = (
    RunnablePassthrough.assign(context=lambda data: data["entry"]["text"])
    | RunnableParallel(
        answer=grounded_prompt | model | text_parser,
        source=RunnableLambda(lambda data: data["entry"]["source"]),
    )
)
rag_chain = retrieve_step | RunnableBranch(
    (lambda data: data["entry"] is not None, answer_from_entry),
    RunnableLambda(lambda data: {"answer": "馆规里没有查到，请咨询服务台。", "source": None}),
)

found = rag_chain.invoke("我想续借，可以延长几天？")
missing = rag_chain.invoke("周日几点开门？")
print(found["answer"], "来源：", found["source"])
print(missing["answer"])

到期前可续借一次，延长 14 天。 来源： 借阅须知 §1
馆规里没有查到，请咨询服务台。


## 9. 多轮聊天：让“它”指向上一轮的书 · 模型

每次调用都要提供希望模型看到的历史。`MessagesPlaceholder` 在模板中为这些消息留出位置。这里用会话名区分读者，只传最近 3 轮完整问答，并在模型回复成功后保存这一轮。内存中的记录随内核重启而清空。

In [12]:
from langchain_core.messages import HumanMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是图书馆学习搭子。用一两句中文回答。"),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
chat_chain = chat_prompt | model
sessions = {}

def chat(session_id, question):
    history = sessions.setdefault(session_id, [])
    reply = chat_chain.invoke({"history": history[-6:], "question": question})
    history.extend([HumanMessage(content=question), reply])
    return reply.text

print(chat("示例读者", "我今天选的书是《西游记》。"))
print(chat("示例读者", "我刚才选了哪本书？"))

《西游记》很适合今天读！你想从孙悟空大闹天宫开始，还是从师徒四人西天取经读起？
你选的是《西游记》。


In [13]:
# 想自己连续聊天时，把 False 改为 True，只运行本格。输入“退出”结束。
interactive_chat = False
if interactive_chat:
    while True:
        question = input("你：").strip()
        if question == "退出":
            break
        if question:
            print("学习搭子：", chat("我的会话", question))

## 10. 重试与备用流程：目录服务偶尔超时 · 本地

`with_retry` 重新执行同一步；`with_fallbacks` 在指定异常发生时改走备用步骤。把策略包在会超时的查询上，可以避免重复运行整条链。下面用本地函数模拟两种故障，方便看清执行次数。

In [14]:
attempts = {}

def query_catalog(title):
    attempts[title] = attempts.get(title, 0) + 1
    # 教学模拟：《西游记》第二次查询成功；另一本书的查询持续超时。
    if title == "西游记" and attempts[title] >= 2:
        return "目录显示可借。"
    raise TimeoutError("目录查询超时")

query_with_retry = RunnableLambda(query_catalog).with_retry(
    retry_if_exception_type=(TimeoutError,),
    stop_after_attempt=2,              # 含第一次，总共最多尝试两次
    wait_exponential_jitter=False,     # 本地演示省去退避等待
)
catalog_chain = query_with_retry.with_fallbacks(
    [RunnableLambda(lambda title: "暂时查不到库存，请稍后再试。")],
    exceptions_to_handle=(TimeoutError,),
)

for title in ["西游记", "红楼梦"]:
    print(title, catalog_chain.invoke(title), "尝试次数：", attempts[title])

西游记 目录显示可借。 尝试次数： 2
红楼梦 暂时查不到库存，请稍后再试。 尝试次数： 2


## 11. 检查和调试：看见输入、执行路径和中间事件 · 本地

模板缺字段时，先看输入 schema；链越来越长时，看图里的节点和边。`with_config` 能给步骤命名和加标签，`astream_events` 可以观察运行事件。下面只运行本地预处理，没有模型用量。

In [15]:
print("模板需要的字段：", term_chain.get_input_jsonschema()["required"])
graph = term_chain.get_graph()
print("流程节点：", [node.name for node in graph.nodes.values()])

named_prepare = prepare.with_config(run_name="整理术语", tags=["学习示例"])
async for event in named_prepare.astream_events(" 续借 ", version="v2"):
    if event["event"] in ("on_chain_start", "on_chain_end"):
        print(event["event"], event["name"], event["data"])

模板需要的字段： ['term']
流程节点： ['PromptInput', 'ChatPromptTemplate', 'ChatOpenAI', 'StrOutputParser', 'StrOutputParserOutput']
on_chain_start 整理术语 {'input': ' 续借 '}
on_chain_end 整理术语 {'output': {'term': '续借'}}


## 回顾与练习

| 遇到的情况 | 从哪里检查 |
|---|---|
| 模板提示缺少键 | 先打印上一步输出，再与模板所需字段比对。第 1 节需要 `{"term": ...}`，第 2 节才支持字符串。 |
| 回答要等到最后才显示 | 看第 6 节消费的是 `stream` 还是 `invoke`，以及下游是否必须等完整文本。 |
| 多轮聊天答非所问 | 检查会话名和传入的 `history`。第 9 节只把最近 3 轮送进模型。 |
| 重试后模型用量增加 | 每次重试都可能重新请求模型。对最小的故障步骤设置有限次数的重试。 |

**动手验证**

1. 在第 4 节给词表添加“续借”，再调用 `route.invoke("续借")`。观察哪条路径执行。
2. 在第 8 节添加一条含出处的“开放时间”记录，然后提问“请问开放时间？”。检查答案和来源是否对应。
3. 在第 9 节换一个新的会话名追问书名。对比同一会话与新会话能看到哪些消息。

**继续查阅**　[Runnable 接口与组合](https://reference.langchain.com/python/langchain-core/runnables/base/Runnable) · [ChatOpenAI 集成](https://docs.langchain.com/oss/python/integrations/chat/openai)